In [39]:
import musicbrainzngs
from dotenv import find_dotenv, load_dotenv
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

from langchain.prompts import ChatPromptTemplate

In [42]:
load_dotenv(find_dotenv("../../creds/.env"), verbose=True)
musicbrainzngs.set_useragent("Example music app", "0.1", "http://example.com/music")

In [14]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:4B",
    temperature=0,
    verbose=True,
    extract_reasoning=True,
)

In [28]:
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""

    airport_code: str = Field(description="airport code to get weather for")


class ArtistSearch(BaseModel):
    """Call this to get the names of songs by a particular artist"""

    artist_name: str = Field(description="name of artist to look up")
    n: int = Field(description="number of results")

In [79]:
def get_tracks(artist_name: str, n: int):
    """Retrieves information about artist's songs from musicbrains"""
    artist = musicbrainzngs.search_artists(artist=artist_name, type="group").get("artist-list", [None])[0]
    artist_id = artist.get("id", None)
    recordings = musicbrainzngs.search_recordings(arid=artist_id)
    tracks = list({item["title"] for item in recordings["recording-list"]})
    return tracks[:n]

In [ ]:
functions = [
    convert_to_openai_function(WeatherSearch),
    convert_to_openai_function(ArtistSearch),
    convert_to_openai_function(get_tracks),
]

In [81]:
llm_with_tools = llm.bind_tools(tools=functions)

In [86]:
prompt = ChatPromptTemplate.from_messages([("system", "You are a helpful assistant"), ("user", "{input}")])

chain = prompt | llm_with_tools

In [83]:
chain.invoke({"input": "What is the weather in Kyiv today?"})

AIMessage(content='\n\n', additional_kwargs={'reasoning_content': "<think>\nOkay, the user is asking for the weather in Kyiv today. Let me check the tools provided. There's a function called WeatherSearch that takes an airport code. Kyiv's airport code is KIV. So I need to call WeatherSearch with airport_code set to KIV. The other tools are for artist songs, which isn't relevant here. Make sure to use the correct airport code. Let me confirm that KIV is indeed Kyiv's airport. Yes, that's right. So the function call should be WeatherSearch with KIV.\n</think>"}, response_metadata={'model': 'qwen3:4B', 'created_at': '2025-06-25T18:38:01.557440054Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2306040881, 'load_duration': 27505850, 'prompt_eval_count': 307, 'prompt_eval_duration': 41985441, 'eval_count': 138, 'eval_duration': 2228516406, 'model_name': 'qwen3:4B'}, id='run--f666df00-ecb9-43d9-bceb-d02c78a23e61-0', tool_calls=[{'name': 'WeatherSearch', 'args': {'airport_code': 'K

In [73]:
chain_result = chain.invoke({"input": "Name 5 songs of Born of Osiris?"})
args = chain_result.tool_calls[0]["args"]
get_tracks(**args)

['XIV', 'Activated', 'Last Straw', 'Glorious Day', 'Seppuku']

In [84]:
chain.invoke({"input": "Name 5 songs of Born of Osiris?"})

AIMessage(content='\n\n', additional_kwargs={'reasoning_content': '<think>\nOkay, the user is asking for five songs by the band Born of Osiris. Let me check the available tools. There\'s the ArtistSearch function which requires an artist name and a number n. The get_tracks function also needs an artist name and n. The WeatherSearch isn\'t relevant here.\n\nSo, I should use either ArtistSearch or get_tracks. Looking at the descriptions, ArtistSearch says it gets the names of songs by an artist, and get_tracks retrieves information about the artist\'s songs. Both seem similar. But the parameters are the same: artist_name and n. The user wants five songs, so n=5.\n\nI need to make sure the artist name is correctly spelled. The user wrote "Born of Osiris", which is the correct name. So I\'ll call the ArtistSearch function with artist_name "Born of Osiris" and n=5. Alternatively, maybe get_tracks? Wait, the get_tracks function\'s description says it retrieves information about the artist\'s